# NLU Prompt to JSON Converter

Notebook chay tren **Google Colab** (GPU runtime), expose Flask API qua **pinggy** tunnel.

Backend (`nluService.ts`) goi: `POST {COLAB_NLU_URL}/api/nlu/parse`  
- Input : `{ "prompt": "..." }`  
- Output: `{ "slots": {...}, "missingSlots": [...], "confidence": 0.0-1.0 }`

### Yeu cau
- Runtime: **GPU** (T4 tren Colab Free)
- Khong can API key - model load tu HuggingFace
- Pinggy tao tunnel mien phi, khong can dang ky

**Chay tuan tu tung cell, hoac Runtime -> Run all.**

In [ ]:
# 1. Cai dependencies
!pip install flask flask-cors transformers accelerate -q
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q

In [ ]:
# 2. Load Qwen2.5-3B-Instruct tu HuggingFace
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
FLASK_PORT  = 5000

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model (lan dau ~2 phut)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'Model san sang tren: {next(model.parameters()).device}')

In [ ]:
# 3. NLU Parser - nhan prompt, tra ve {slots, missingSlots, confidence}
# Khop voi contract NluParseResponse trong nluService.ts
import json, re
from datetime import date

_YEAR = date.today().year

# Danh sach 10 tag duy nhat ton tai trong DB (xem backend/src/scripts/seed-places.ts)
_VALID_TAGS = {
    'beach', 'mountain', 'culture', 'food', 'spiritual',
    'shopping', 'entertainment', 'nature', 'sport', 'landmark',
}

# Stop-words/verbs khong nen xuat hien trong experienceKeywords
_EXP_STOPWORDS = {
    'an', 'di', 'xem', 'thich', 'muon', 'duoc', 'co', 'la', 'va', 'voi',
    'choi', 'tham', 'ghe', 'toi', 'minh', 'cung',
}

_SYSTEM = (
    'You are an NLU system for a Vietnamese travel planning app.\n'
    'Extract slots from the user prompt and return ONLY a valid JSON object.\n'
    'No explanation, no markdown, no code block — raw JSON only.\n\n'
    'JSON schema:\n'
    '{\n'
    '  "slots": {\n'
    '    "destinationCity":      string or null,\n'
    '    "durationDays":         integer or null,\n'
    '    "startDate":            "YYYY-MM-DD" or null,\n'
    '    "preferredTagNames":    string[],\n'
    '    "experienceKeywords":   string[],\n'
    '    "budgetTotal":          number (VND) or null,\n'
    '    "groupType":            "solo"|"couple"|"family"|"friends"|"business" or null,\n'
    '    "mobilityRestrictions": string[],\n'
    '    "dietaryPreferences":   string[],\n'
    '    "pace":                 integer 1-5 or null\n'
    '  },\n'
    '  "missingSlots": string[],\n'
    '  "confidence":   float 0.0-1.0\n'
    '}\n\n'
    'Extraction rules:\n'
    '- destinationCity: Vietnamese city/province name as-is (e.g. "Da Lat", "Hoi An")\n'
    '- durationDays: number of travel days (integer)\n'
    f'- startDate: YYYY-MM-DD; "1/5" -> "{_YEAR}-05-01"; null if not mentioned\n'
    '- preferredTagNames: ONLY these 10 lowercase tags are allowed.\n'
    '    NEVER invent tags outside this list. Return [] if nothing fits.\n'
    '    bien/bai bien -> beach\n'
    '    nui/leo nui/trekking -> mountain\n'
    '    van hoa/lich su/bao tang/di san/pho co -> culture\n'
    '    am thuc/an uong/mon an/ca phe/quan an/nha hang -> food\n'
    '    chua/den/tam linh/nha tho -> spiritual\n'
    '    mua sam/cho dem/cho -> shopping\n'
    '    giai tri/cong vien giai tri/show/quan bar -> entertainment\n'
    '    thien nhien/thac nuoc/cong vien quoc gia/rung/ho -> nature\n'
    '    the thao/lan bien/dap xe/chay bo/cam trai/leo nui the thao -> sport\n'
    '    diem tham quan/landmark/cau/quang truong/check-in -> landmark\n'
    '- experienceKeywords: array of free-text Vietnamese phrases describing\n'
    '    SPECIFIC experiences NOT covered by the tag list above.\n'
    '    Keep diacritics, lowercase, 1-4 words per phrase. Max 8 phrases.\n'
    '    Skip pure verbs ("an", "di", "xem", "thich"); extract noun phrases.\n'
    '    Examples:\n'
    '      "đi dạo dưới ánh hoàng hôn" -> ["hoàng hôn"]\n'
    '      "ăn mực nướng" -> ["mực nướng"]\n'
    '      "ăn bánh mì chả cá" -> ["bánh mì chả cá"]\n'
    '      "ngắm bình minh trên núi" -> ["bình minh"]\n'
    '      "chụp ảnh đèn lồng phố cổ" -> ["đèn lồng"]\n'
    '    Return [] if none.\n'
    '- budgetTotal: total VND ("5 trieu"->5000000, "500k"->500000)\n'
    '- groupType: mot minh->solo, doi/cap/nguoi yeu/vo chong->couple,\n'
    '             gia dinh->family, ban be/nhom->friends, cong tac->business\n'
    '- mobilityRestrictions: accessibility needs (e.g. ["wheelchair"]); [] if none\n'
    '- dietaryPreferences: an chay->vegetarian, thuan chay->vegan, halal->halal; [] if none\n'
    '- pace: 1=very relaxed, 2=relaxed, 3=normal, 4=active, 5=packed; null if not mentioned\n'
    '- missingSlots: names of slots with null or [] value\n'
    '- confidence: 0.0-1.0\n\n'
    'Example:\n'
    'Input: "Đi Đà Nẵng 3 ngày với gia đình, ngân sách 5 triệu, ăn bánh mì chả cá và đi dạo dưới hoàng hôn"\n'
    'Output: {"slots":{"destinationCity":"Đà Nẵng","durationDays":3,"startDate":null,'
    '"preferredTagNames":["food"],"experienceKeywords":["bánh mì chả cá","hoàng hôn"],'
    '"budgetTotal":5000000,"groupType":"family",'
    '"mobilityRestrictions":[],"dietaryPreferences":[],"pace":null},'
    '"missingSlots":["startDate","mobilityRestrictions","dietaryPreferences","pace"],'
    '"confidence":0.92}'
)


def _empty_slots():
    return {
        'destinationCity':      None,
        'durationDays':         None,
        'startDate':            None,
        'preferredTagNames':    [],
        'experienceKeywords':   [],
        'budgetTotal':          None,
        'groupType':            None,
        'mobilityRestrictions': [],
        'dietaryPreferences':   [],
        'pace':                 None,
    }


def _normalise(raw):
    s = _empty_slots()
    GROUPS = {'solo', 'couple', 'family', 'friends', 'business'}

    s['destinationCity'] = raw.get('destinationCity') or None

    dur = raw.get('durationDays')
    s['durationDays'] = int(dur) if isinstance(dur, (int, float)) and dur > 0 else None

    sd = raw.get('startDate')
    s['startDate'] = sd if isinstance(sd, str) and sd else None

    # Constrain preferredTagNames to the 10 valid DB tags
    tags = raw.get('preferredTagNames', [])
    if isinstance(tags, list):
        s['preferredTagNames'] = [
            t.strip().lower() for t in tags
            if isinstance(t, str) and t.strip().lower() in _VALID_TAGS
        ]

    # Free-text experience keywords with stop-word filter and cap
    exp = raw.get('experienceKeywords', [])
    if isinstance(exp, list):
        cleaned = []
        seen = set()
        for e in exp:
            if not isinstance(e, str):
                continue
            v = e.strip().lower()
            if not v:
                continue
            # Loai bo cum chi co stop-words (vd "an", "di xem")
            tokens = [w for w in re.split(r'\s+', v) if w]
            non_stop = [w for w in tokens if w not in _EXP_STOPWORDS]
            if not non_stop:
                continue
            phrase = ' '.join(tokens)
            if phrase not in seen:
                seen.add(phrase)
                cleaned.append(phrase)
        s['experienceKeywords'] = cleaned[:8]

    budget = raw.get('budgetTotal')
    s['budgetTotal'] = float(budget) if isinstance(budget, (int, float)) and budget > 0 else None

    gt = raw.get('groupType')
    s['groupType'] = gt if gt in GROUPS else None

    mob = raw.get('mobilityRestrictions', [])
    s['mobilityRestrictions'] = [m for m in mob if isinstance(m, str)] if isinstance(mob, list) else []

    diet = raw.get('dietaryPreferences', [])
    s['dietaryPreferences'] = [d for d in diet if isinstance(d, str)] if isinstance(diet, list) else []

    pace = raw.get('pace')
    s['pace'] = int(pace) if isinstance(pace, (int, float)) and 1 <= pace <= 5 else None

    return s


def _extract_json(text):
    # Tim JSON object dau tien trong output cua model
    brace = text.find('{')
    if brace == -1:
        raise ValueError(f'No JSON found in: {text[:300]}')
    depth, end = 0, brace
    for i, ch in enumerate(text[brace:], brace):
        if ch == '{': depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i
                break
    return text[brace:end+1]


def parse_prompt(prompt):
    messages = [
        {'role': 'system', 'content': _SYSTEM},
        {'role': 'user',   'content': f'Parse this travel prompt:\n{prompt}'},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,           # greedy - on dinh hon voi JSON
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out_ids[0][inputs['input_ids'].shape[1]:]
    raw_text   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    raw    = json.loads(_extract_json(raw_text))
    slots  = _normalise(raw.get('slots', {}))
    # experienceKeywords la field tuy chon - khong tinh vao missingSlots
    missing = [
        k for k, v in slots.items()
        if (v is None or v == []) and k != 'experienceKeywords'
    ]
    conf   = float(raw.get('confidence', 0.8))

    return {
        'slots':        slots,
        'missingSlots': missing,
        'confidence':   round(max(0.0, min(1.0, conf)), 4),
    }


print('NLU parser ready')

In [ ]:
# 4. Flask API - expose endpoint /api/nlu/parse cho backend
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'service': 'nlu-qwen'})


@app.route('/api/nlu/parse', methods=['POST'])
def nlu_parse():
    data = request.get_json(silent=True)
    if not data:
        return jsonify({'error': 'REQUEST_BODY_MISSING'}), 400

    prompt = data.get('prompt', '')
    if not isinstance(prompt, str) or not prompt.strip():
        return jsonify({'error': 'PROMPT_IS_EMPTY'}), 400

    try:
        return jsonify(parse_prompt(prompt.strip()))
    except json.JSONDecodeError as e:
        return jsonify({'error': f'MODEL_OUTPUT_NOT_JSON: {e}'}), 502
    except Exception as e:
        return jsonify({'error': str(e)}), 500


print('Flask app defined')

In [ ]:
# 5. Khoi dong pinggy tunnel + Flask server
# Pinggy tao HTTPS tunnel mien phi, khong can tai khoan
import subprocess, threading, re, time

public_url = None


def _start_pinggy():
    global public_url
    proc = subprocess.Popen(
        ['ssh', '-p', '443',
         '-R', f'0:localhost:{FLASK_PORT}',
         '-o', 'StrictHostKeyChecking=no',
         '-o', 'ServerAliveInterval=30',
         'a.pinggy.io'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print('[pinggy]', line.rstrip())
        m = re.search(r'https?://[\w.-]+\.pinggy\.link', line)
        if m:
            public_url = m.group()
            break


pinggy_thread = threading.Thread(target=_start_pinggy, daemon=True)
pinggy_thread.start()

# Doi pinggy lay URL (toi da 15 giay)
for _ in range(30):
    if public_url:
        break
    time.sleep(0.5)

if not public_url:
    print('WARN: Chua lay duoc public URL, kiem tra output pinggy o tren.')
else:
    print('=' * 60)
    print(f'NLU Server san sang')
    print(f'Public URL : {public_url}')
    print(f'Endpoint   : {public_url}/api/nlu/parse')
    print()
    print('Copy dong nay vao .env cua backend:')
    print(f'  COLAB_NLU_URL={public_url}')
    print('=' * 60)


def _run_flask():
    app.run(host='0.0.0.0', port=FLASK_PORT, use_reloader=False)


threading.Thread(target=_run_flask, daemon=True).start()

In [ ]:
# 6. Test thu cong (chay sau Cell 5)
import requests, json, time

time.sleep(1)
LOCAL = f'http://localhost:{FLASK_PORT}'

cases = [
    'Toi muon di Da Lat 3 ngay voi gia dinh, ngan sach 5 trieu, thich ca phe va thac nuoc',
    'Du lich Hoi An 5 ngay tu ngay 1/5, di cung nguoi yeu, budget 10 trieu, an chay, di thu thai',
    'Mot minh kham pha Ha Noi 2 ngay cuoi tuan, thich am thuc duong pho va van hoa',
    'Di Phu Quoc 7 ngay voi nhom ban, thich lan bien va chup anh, pace nhanh',
    # Cases test trich xuat experienceKeywords (cum mo ta trai nghiem)
    'Đi Đà Nẵng 3 ngày, ăn bánh mì chả cá và ngắm hoàng hôn ở biển',
    'Hội An 2 ngày, đi dạo phố cổ buổi tối, thích chụp ảnh đèn lồng',
]

for p in cases:
    label = (p[:70] + '...') if len(p) > 70 else p
    print(f'\nPrompt: {label}')
    r = requests.post(f'{LOCAL}/api/nlu/parse', json={'prompt': p}, timeout=60)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))
    print('-' * 60)